# Tokenizador 

In [13]:
from pathlib import Path
import sys

path = Path().resolve().parents[1]

if str(path) not in sys.path:
    sys.path.append(str(path))

In [14]:
from src.helper_functions import load_data

data = load_data()

data_len = data.shape[0]
train_data = data.iloc[:int(data_len * 0.6)]

In [15]:
train_text = []
for pt_s, en_s in zip(train_data.pt_sentences, train_data.en_sentences):
    train_text.append(pt_s)
    train_text.append(en_s)

In [16]:
import tokenizers
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer

#1. Criando o tokenizador
tokenizer = tokenizers.Tokenizer(
    BPE(unk_token="<unk>")
)

#2. Pré-Tokenização
tokenizer.pre_tokenizer = Whitespace()

#3. Criando objeto de trainamento

special_tokens = ["<pad>", "<unk>", "<bos>", "<eos>"]
bpe_trainer = BpeTrainer(
    vocab_size=8000,
    special_tokens=special_tokens
)

#4. Treinando o Modelo Efetivamente
tokenizer.train_from_iterator(
    iterator=train_text,
    trainer=bpe_trainer
)

#5. Habilitando Padding
tokenizer.enable_padding(
    pad_id=0,
    pad_token="<pad>"
)

#6. Habilitar Truncation
tokenizer.enable_truncation(
    max_length=128
)

In [17]:
exemplo = train_data.en_sentences[0]
bpe_encoding = tokenizer.encode(exemplo)

ids = bpe_encoding.ids

bpe_decoding = tokenizer.decode(ids)
bpe_decoding

"Let ' s try something ."

In [39]:
tokenizer_path = path / "data" / "tokenizer"

if not tokenizer_path.exists():
    tokenizer_path.mkdir()

path = tokenizer_path / 'en-pt-cased-pont.json'

tokenizer.save(
    path= str(path)
)